# dev/camera — exposure and white balance, measured on the disc

The cameras' own auto exposure meters the **whole frame**, so the white disc on a darker background is pushed to full white. This notebook measures each camera **on the disc itself** and proposes the numbers for its `camera_cfg`.

For one camera it:
1. reads the camera's exposure, gain and white balance as they are now;
2. finds the disc with apc's own detector (`vision/disc_od.yaml`). Its **face** is the circle `FACE_SHARE` of the detector box, inside the band;
3. pins gain at `GAIN` (dB), takes one picture at each exposure of `SWEEP` (µs), and measures the face: its median brightness and the share of it that is clipped;
4. picks the exposure whose face median is nearest `TARGET` with under `MAX_CLIPPED` % clipped, then lets white balance settle **once** on the disc and reads the ratios back;
5. **puts the camera back exactly as step 1 found it**, whatever happens.

It moves nothing. Everything goes through the vision server, at the address in `scene/bench.j2`. The only thing it writes is the sweep's pictures, to `captures/exposure/<time>_<camera>/` (git-ignored). The proposal is printed; copying it into `scene/bench.j2` and `scene/layout.j2`'s `camera_cfg` is yours.

Run **Setup** once, then the cell of the camera that has a disc in view.

## Setup

In [ ]:
import time
from pathlib import Path

import cv2
import numpy as np
from IPython.display import Image, display
from jinja2 import Template
from dorna_vision_client import VisionClient

# Walk up to the project root (the folder holding launch.yaml).
PROJ = Path.cwd()
while not (PROJ / "launch.yaml").exists():
    PROJ = PROJ.parent

CAMERAS = {"bottom": "camera_serial_station", "top": "camera_serial_robot"}
SWEEP = [60000, 45000, 35000, 27000, 20000, 15000, 11000, 8000, 6000]   # µs
GAIN = 0.0            # dB, pinned for the sweep — 0 adds no noise
TARGET = 195          # face median to aim for, 0..255 — room above for the band's glare
MAX_CLIPPED = 1.0     # % of the face allowed at CLIP or over
CLIP = 250            # a channel at or over this is clipped (JPEG rounds 255 down a little)
FACE_SHARE = 0.80     # the face: this share of the detector box's half-side, inside the band
WB_FRAMES = 6         # pictures for a "once" white balance to settle on the disc
QUALITY = 95          # JPEG quality of the measured pictures

BENCH = Template((PROJ / "scene" / "bench.j2").read_text()).module   # this unit's addresses and serials


def grab(vc, sn):
    jpeg, _meta = vc.camera_get_img(sn, quality=QUALITY)
    return cv2.imdecode(np.frombuffer(jpeg, np.uint8), cv2.IMREAD_COLOR)


def face_circle(box):
    xs = [c[0] for c in box]
    ys = [c[1] for c in box]
    cx, cy = (min(xs) + max(xs)) / 2, (min(ys) + max(ys)) / 2
    return round(cx), round(cy), round(FACE_SHARE * min(max(xs) - min(xs), max(ys) - min(ys)) / 2)


def measure(img, circle):
    """The face's median brightness (its brightest channel) and the % of it
    clipped in any channel."""
    mask = np.zeros(img.shape[:2], np.uint8)
    cv2.circle(mask, circle[:2], circle[2], 255, -1)
    top = img.max(axis=2)[mask > 0]
    return float(np.median(top)), 100.0 * float((top >= CLIP).mean())


def crop(img, circle, side=220):
    cx, cy, r = circle
    r = int(r / FACE_SHARE)
    c = img[max(0, cy - r):cy + r, max(0, cx - r):cx + r]
    return cv2.resize(c, (side, side))


def restore(vc, sn, before):
    """The camera back as `before` (camera_info) found it."""
    exp, gain, wb = before["exposure"], before["gain"], before["wb"]
    if exp["auto"]:
        vc.camera_exposure(sn, auto=True)
    else:
        vc.camera_exposure(sn, us=exp["us"])
    if gain["auto"]:
        vc.camera_gain(sn, auto=True)
    else:
        vc.camera_gain(sn, db=gain["db"])
    if wb["cfg"].get("auto"):
        vc.camera_wb(sn, auto=True)
    else:
        vc.camera_wb(sn, ratios=wb["ratios"])


def measure_camera(camera, gain=GAIN, sweep=SWEEP):
    """Sweep one camera ("bottom" / "top") on the disc in view, show the
    face at every exposure, print the proposal, put the camera back."""
    sn = str(getattr(BENCH, CAMERAS[camera]))
    out = PROJ / "captures" / "exposure" / f"{time.strftime('%Y-%m-%d_%H-%M-%S')}_{camera}"
    out.mkdir(parents=True, exist_ok=True)

    vc = VisionClient()
    vc.connect(host=BENCH.vision_ip, port=int(BENCH.vision_port))
    before = vc.camera_info(sn)
    for k in ("exposure", "gain", "wb"):
        if k not in before or "error" in before[k]:
            vc.close()
            raise RuntimeError(f"{camera} camera {sn}: the vision server reports no {k} "
                               f"({before.get(k)}) — upgrade the vision unit first")
    print(f"{camera} camera {sn} now: exposure {before['exposure']}, gain {before['gain']}, wb {before['wb']}")

    det = f"exposure_{camera}"
    vc.detection_add(det, camera_serial_number=sn, config=str(PROJ / "vision" / "disc_od.yaml"))
    try:
        found = vc.detection_run(det)
        if not found:
            raise RuntimeError(f"no disc found by the {camera} camera — put one where the run inspects it")
        best = max(found, key=lambda r: r.get("conf", 0))
        circle = face_circle(best["corners"])
        print(f"disc found, conf {best.get('conf', 0):.2f}")

        vc.camera_gain(sn, db=gain)
        rows, crops = [], []
        print(f"\n{'exposure µs':>12} {'face median':>12} {'clipped %':>10}")
        for us in sweep:
            vc.camera_exposure(sn, us=us)
            grab(vc, sn)                                   # the first at the new setting, discarded
            img = grab(vc, sn)
            med, clipped = measure(img, circle)
            cv2.imwrite(str(out / f"exposure_{us}.jpg"), img)
            rows.append((us, med, clipped))
            crops.append(crop(img, circle))
            print(f"{us:>12} {med:>12.0f} {clipped:>10.1f}")
        display(Image(data=cv2.imencode(".jpg", np.hstack(crops))[1].tobytes()))

        usable = [r for r in rows if r[2] < MAX_CLIPPED]
        if not usable:
            raise RuntimeError(f"every exposure clips over {MAX_CLIPPED} % of the face — try a shorter sweep")
        us, med, clipped = min(usable, key=lambda r: abs(r[1] - TARGET))

        vc.camera_exposure(sn, us=us)
        vc.camera_wb(sn, once=True)
        for _ in range(WB_FRAMES):
            grab(vc, sn)
        ratios = vc.camera_wb(sn)["wb"]["ratios"]
        chosen = grab(vc, sn)
        cv2.imwrite(str(out / f"chosen_{us}.jpg"), chosen)
    finally:
        vc.detection_remove(det)
        restore(vc, sn, before)
        vc.close()

    print(f"\nthe chosen exposure, white balance settled on the disc:")
    display(Image(data=cv2.imencode(".jpg", crop(chosen, circle, 400))[1].tobytes()))
    print(f"pictures: {out}")
    print(f"camera put back as it was: exposure {before['exposure']}, gain {before['gain']}")
    print(f"\nproposed for the {camera} camera {sn} — face median {med:.0f} (target {TARGET}), {clipped:.1f} % clipped:")
    print(f"  exposure: {us}        # µs")
    print(f"  gain: {gain:g}             # dB")
    print(f"  wb: {{ratios: {{red: {ratios['red']}, green: {ratios['green']}, blue: {ratios['blue']}}}}}")
    if abs(med - TARGET) > 10:
        print(f"  (no sweep step landed within 10 of {TARGET} — call again with a narrower sweep around {us})")
    return {"exposure": us, "gain": gain, "wb": {"ratios": ratios}, "median": med, "clipped": clipped}

## Bottom camera

A disc held in the gripper at the station camera: the run paused at **inspecting**, or `camera.ipynb`'s present cell.

In [ ]:
bottom = measure_camera("bottom")

## Top camera

A disc on the anode, the robot at its viewing stand: the run paused at **inspecting on the anode**.

In [ ]:
top = measure_camera("top")